In [ ]:
import pandas as pd
import os
os.chdir(r'C:\Users\Sparsh Kapoor\OneDrive\Desktop\udemy data_analyst bootcamp\python\upi_project')
#  to change the directory to the folder where the csv file is located.

In [14]:
# Step 1: Load the file 
df_raw = pd.read_excel(
      'RBIB Table No. 45 _ Payment System Indicators.xlsx',
    header=None
)
print('File loaded successfully ✅')
print('Shape (rows x columns):', df_raw.shape)

# We are using header =none becuase as there are multiple header rows in the data file , we will ask the pandas
# to import as just raw plain data  and then we will do the cleaning and formatting of the data as per our requirements.



File loaded successfully ✅
Shape (rows x columns): (88, 142)


c:\Users\Sparsh Kapoor\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [ ]:
df_raw.iloc[:8, :5]
# first 8 rows (rows 0 to 7) and the first 5 columns (columns 0 to 4) of the dataframe
# Understanding the data : we are looking at the first 8 rows and first 5 columns of the data to understand how the data is structured and what kind of cleaning and formatting we need to do to make it ready for analysis.

,0,1,2,3,4
0,NaN,NaN,NaN,NaN,NaN
1,NaN,Payment System Indicators,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,Month/Year,A. Settlement Systems,NaN,NaN
4,NaN,NaN,Financial Market Infrastructures (FMIs),NaN,NaN
5,NaN,NaN,1 CCIL Operated Systems (1.1 to 1.3),NaN,1.1 Govt. Securities Clearing (1.1.1 to 1.1.3)
6,NaN,NaN,Volume (Lakh),Value( Rupees Crores ),Volume (Lakh)
7,NaN,Mar-2026,4.438,30819254.479,1.512


In [ ]:
#Rows 0–6 are all header rows (title, categories, sub-categories, Volume/Value labels)
#Actual data starts at row 7**
#Column 1 = Month/Year (e.g. Mar-2026)
#Column 36 = UPI Volume** (in Lakh transactions)
#Column 37 = UPI Value** (in ₹ Crores)

In [16]:
# Grab only the 3 columns we need, starting from row 7 (where data begins)
df = df_raw.iloc[7:, [1, 36, 37]].copy()

# Renaming the extracted column 
df.columns = ['month_year', 'upi_volume_lakh', 'upi_value_crore']

# Reset the index
df = df.reset_index(drop=True)
# It will reset the index of the dataframe and drop the old index. 
# This is important because after we have extracted the relevant columns and rows, the index will still be based on the original dataframe, which may not be sequential or start from 0. 
# By resetting the index, we will have a clean and sequential index for our new dataframe.

print('Columns extracted ✅')
df.head(10)

Columns extracted ✅


,month_year,upi_volume_lakh,upi_value_crore
0,Mar-2026,226411.08,2952542.03
1,Feb-2026,203941.84,2684229.3
2,Jan-2026,217034.45,2833481.26
3,Dec-2025,216346.68,2796712.71
4,Nov-2025,204669.79,2631632.64
5,Oct-2025,207009.21,2727790.71
6,Sep-2025,196334.33,2489736.52
7,Aug-2025,200083.1,2485472.9
8,Jul-2025,194679.48,2508498.1
9,Jun-2025,183950.06,2403930.69


In [ ]:
# Keep only rows where month_year looks like 'Nov-2019' (starts with 3 letters)
df = df[df['month_year'].apply(lambda x: isinstance(x, str) and x[:3].isalpha())]

# Reset index again after removing rows
df = df.reset_index(drop=True)

# Display the len of the rows and to verify the filtering has worked correctly , printing the first and last value
print(f'Clean rows remaining: {len(df)}')
print(f'First entry : {df["month_year"].iloc[0]}')
print(f'Last entry  : {df["month_year"].iloc[-1]}')

Clean rows remaining: 77
First entry : Mar-2026
Last entry  : Nov-2019


In [ ]:
df['upi_volume_lakh'] = pd.to_numeric(df['upi_volume_lakh'], errors='coerce')
df['upi_value_crore'] = pd.to_numeric(df['upi_value_crore'], errors='coerce')

print('Data types now:')
print(df.dtypes)

# Making sure that the data types of the columns are numeric 

Data types now:
month_year          object
upi_volume_lakh    float64
upi_value_crore    float64
dtype: object


In [20]:
# Convert to proper date format usind the pandas datetime function and specifying the format of the date in the original data which is month-year format (e.g. Mar-2019)
df['date'] = pd.to_datetime(df['month_year'], format='%b-%Y')

# Add helper columns
df['year']       = df['date'].dt.year
df['month_num']  = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%B')
df['quarter']    = 'Q' + df['date'].dt.quarter.astype(str)


df[['month_year', 'date', 'year', 'month_num', 'month_name', 'quarter']].head(8)

,month_year,date,year,month_num,month_name,quarter
0,Mar-2026,2026-03-01,2026,3,March,Q1
1,Feb-2026,2026-02-01,2026,2,February,Q1
2,Jan-2026,2026-01-01,2026,1,January,Q1
3,Dec-2025,2025-12-01,2025,12,December,Q4
4,Nov-2025,2025-11-01,2025,11,November,Q4
5,Oct-2025,2025-10-01,2025,10,October,Q4
6,Sep-2025,2025-09-01,2025,9,September,Q3
7,Aug-2025,2025-08-01,2025,8,August,Q3


In [21]:
# Formula: (Value in Crore × 10,000,000) ÷ (Volume in Lakh × 100,000)
# = Value in ₹ ÷ number of transactions = ₹ per transaction
df['avg_txn_value_rs'] = ((df['upi_value_crore'] * 1e7) / (df['upi_volume_lakh'] * 1e5)).round(2)


df[['month_year', 'upi_volume_lakh', 'upi_value_crore', 'avg_txn_value_rs']].head()

,month_year,upi_volume_lakh,upi_value_crore,avg_txn_value_rs
0,Mar-2026,226411.08,2952542.03,1304.06
1,Feb-2026,203941.84,2684229.30,1316.17
2,Jan-2026,217034.45,2833481.26,1305.54
3,Dec-2025,216346.68,2796712.71,1292.70
4,Nov-2025,204669.79,2631632.64,1285.79


In [ ]:
# Sort oldest first on the basis of date column and reset index again
df = df.sort_values('date').reset_index(drop=True)


print(f'Total rows    : {len(df)}')
print(f'Date range    : {df["month_year"].iloc[0]}  to  {df["month_year"].iloc[-1]}')
print(f'Missing values: {df.isnull().sum().sum()}')
print()
df.head()

=== FINAL DATA SUMMARY ===
Total rows    : 77
Date range    : Nov-2019  to  Mar-2026
Missing values: 0



,month_year,upi_volume_lakh,upi_value_crore,date,year,month_num,month_name,quarter,avg_txn_value_rs
0,Nov-2019,12187.70742,189229.089108,2019-11-01,2019,11,November,Q4,1552.62
1,Dec-2019,13084.01713,202520.758566,2019-12-01,2019,12,December,Q4,1547.85
2,Jan-2020,13050.19232,216242.974143,2020-01-01,2020,1,January,Q1,1657.01
3,Feb-2020,13256.93237,222516.950030,2020-02-01,2020,2,February,Q1,1678.50
4,Mar-2020,12468.44550,206462.307026,2020-03-01,2020,3,March,Q1,1655.88


In [23]:
# Quick look at the full cleaned table
df

,month_year,upi_volume_lakh,upi_value_crore,date,year,month_num,month_name,quarter,avg_txn_value_rs
0,Nov-2019,12187.70742,1.892291e+05,2019-11-01,2019,11,November,Q4,1552.62
1,Dec-2019,13084.01713,2.025208e+05,2019-12-01,2019,12,December,Q4,1547.85
2,Jan-2020,13050.19232,2.162430e+05,2020-01-01,2020,1,January,Q1,1657.01
3,Feb-2020,13256.93237,2.225170e+05,2020-02-01,2020,2,February,Q1,1678.50
4,Mar-2020,12468.44550,2.064623e+05,2020-03-01,2020,3,March,Q1,1655.88
...,...,...,...,...,...,...,...,...,...
72,Nov-2025,204669.79000,2.631633e+06,2025-11-01,2025,11,November,Q4,1285.79
73,Dec-2025,216346.68000,2.796713e+06,2025-12-01,2025,12,December,Q4,1292.70
74,Jan-2026,217034.45000,2.833481e+06,2026-01-01,2026,1,January,Q1,1305.54
75,Feb-2026,203941.84000,2.684229e+06,2026-02-01,2026,2,February,Q1,1316.17


In [24]:
df.to_csv('upi_clean_data.csv', index=False)

print('✅ File saved as upi_clean_data.csv')
print(f'   Rows    : {len(df)}')
print(f'   Columns : {list(df.columns)}')

✅ File saved as upi_clean_data.csv
   Rows    : 77
   Columns : ['month_year', 'upi_volume_lakh', 'upi_value_crore', 'date', 'year', 'month_num', 'month_name', 'quarter', 'avg_txn_value_rs']
